# 05 — Revenue Opportunity Analysis
## But First, Coffee: Competitive Revenue Intelligence

This notebook translates the competitive, pricing, promotional, and customer-experience evidence into commercially relevant revenue opportunities for But First, Coffee.

The analysis focuses on identifying potential revenue levers while preserving the brand's observed affordability position.

Because internal transaction, product-mix, cost, and contribution-margin data are unavailable, the scenarios presented here are not forecasts of actual revenue or profitability. They are decision-support scenarios showing how changes in price, volume, basket size, promotions, and execution could influence revenue under stated assumptions.

## Revenue Management Framework

The analysis is structured around the basic revenue relationship:

\[
Revenue = Transactions \times Average\ Transaction\ Value
\]

Average Transaction Value can be decomposed as:

\[
ATV = Average\ Selling\ Price \times Items\ per\ Transaction
\]

This creates three primary external revenue levers:

1. **Transaction Volume**
   - customer acquisition;
   - repeat visits;
   - channel availability;
   - branch accessibility;
   - service and order execution.

2. **Average Selling Price**
   - selective pricing;
   - product premiumization;
   - size upgrades;
   - premium variants;
   - promotion optimization.

3. **Items per Transaction**
   - food attachment;
   - add-ons;
   - bundles;
   - cross-selling;
   - complementary products.

With internal cost data, the model should subsequently incorporate:

\[
Contribution\ Margin
=
Selling\ Price - Variable\ Cost
\]

and:

\[
Contribution\ Margin\%
=
\frac{Contribution\ Margin}{Selling\ Price}
\times100
\]

Therefore, the present analysis identifies **revenue and profitability opportunities**, but does not estimate actual profitability.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent

DATA_CLEANED = (
    PROJECT_ROOT
    / "data"
    / "cleaned"
)

TABLES = (
    PROJECT_ROOT
    / "outputs"
    / "tables"
)

FIGURES = (
    PROJECT_ROOT
    / "outputs"
    / "figures"
)

print(
    "Project root:",
    PROJECT_ROOT
)

print(
    "Revenue opportunity analysis ready."
)

Project root: /Users/jannoelvero/Desktop/but_first_coffee_revenue_analysis
Revenue opportunity analysis ready.


In [2]:
menu_prices = pd.read_csv(
    DATA_CLEANED
    / "menu_prices_cleaned.csv"
)

matched_prices = pd.read_csv(
    DATA_CLEANED
    / "matched_family_prices.csv"
)

reviews = pd.read_csv(
    DATA_CLEANED
    / "reviews_cleaned.csv"
)

channels = pd.read_csv(
    DATA_CLEANED
    / "channels_partnerships_cleaned.csv"
)

statistical_summary = pd.read_csv(
    TABLES
    / "statistical_analysis_summary.csv"
)

print(
    "Menu prices:",
    menu_prices.shape
)

print(
    "Matched prices:",
    matched_prices.shape
)

print(
    "Reviews:",
    reviews.shape
)

print(
    "Channels:",
    channels.shape
)

print(
    "Statistical summary:",
    statistical_summary.shape
)

Menu prices: (150, 16)
Matched prices: (15, 4)
Reviews: (282, 52)
Channels: (16, 6)
Statistical summary: (7, 4)


In [3]:
datasets = {
    "menu_prices": menu_prices,
    "matched_prices": matched_prices,
    "reviews": reviews,
    "channels": channels,
    "statistical_summary": statistical_summary
}

for name, df in datasets.items():

    print(
        f"\n{name.upper()}"
    )

    print(
        df.columns.tolist()
    )


MENU_PRICES
['brand', 'branch', 'category', 'product', 'regular_price_php', 'promo_price_php', 'platform', 'source_url', 'is_promo', 'effective_price_php', 'discount_php', 'discount_pct', 'category_original', 'category_standard', 'product_class', 'comparable_product_family']

MATCHED_PRICES
['brand', 'comparable_product_family', 'representative_price_php', 'source_observations']

REVIEWS
['brand', 'branch', 'platform', 'aggregate_rating', 'rating_volume', 'review_date', 'review_text', 'initial_theme_hint', 'source_url', 'review_text_original', 'theme_original', 'review_text_clean', 'theme_list', 'review_date_parsed', 'theme_standard_list', 'theme_add_on', 'theme_availability', 'theme_coffee_strength', 'theme_consistency', 'theme_customization', 'theme_delivery', 'theme_food_quality', 'theme_loyalty', 'theme_missing_add_on', 'theme_missing_item', 'theme_occasion', 'theme_order_accuracy', 'theme_order_identification', 'theme_packaging', 'theme_packaging_accessories', 'theme_portion', 't

In [4]:
matched_price_matrix = (
    matched_prices
    .pivot(
        index="comparable_product_family",
        columns="brand",
        values="representative_price_php"
    )
)

matched_price_matrix

brand,"But First, Coffee",Starbucks,The Coffee Bean & Tea Leaf
comparable_product_family,,,
Americano,130.0,175.0,185.0
Cafe Latte,150.0,185.0,187.5
Caramel Macchiato,130.0,210.0,245.0
Matcha Latte,130.0,190.0,225.0
Mocha,150.0,205.0,220.0


In [5]:
pricing_headroom = pd.DataFrame({
    "product_family":
        matched_price_matrix.index,

    "bfc_price":
        matched_price_matrix[
            "But First, Coffee"
        ].values,

    "starbucks_price":
        matched_price_matrix[
            "Starbucks"
        ].values,

    "cbtl_price":
        matched_price_matrix[
            "The Coffee Bean & Tea Leaf"
        ].values
})

pricing_headroom[
    "gap_vs_starbucks_php"
] = (
    pricing_headroom[
        "starbucks_price"
    ]
    - pricing_headroom[
        "bfc_price"
    ]
)

pricing_headroom[
    "gap_vs_cbtl_php"
] = (
    pricing_headroom[
        "cbtl_price"
    ]
    - pricing_headroom[
        "bfc_price"
    ]
)

pricing_headroom[
    "bfc_discount_vs_starbucks_pct"
] = (
    pricing_headroom[
        "gap_vs_starbucks_php"
    ]
    / pricing_headroom[
        "starbucks_price"
    ]
    * 100
)

pricing_headroom[
    "bfc_discount_vs_cbtl_pct"
] = (
    pricing_headroom[
        "gap_vs_cbtl_php"
    ]
    / pricing_headroom[
        "cbtl_price"
    ]
    * 100
)

pricing_headroom.round(2)

,product_family,bfc_price,starbucks_price,cbtl_price,gap_vs_starbucks_php,gap_vs_cbtl_php,bfc_discount_vs_starbucks_pct,bfc_discount_vs_cbtl_pct
0,Americano,130.0,175.0,185.0,45.0,55.0,25.71,29.73
1,Cafe Latte,150.0,185.0,187.5,35.0,37.5,18.92,20.00
2,Caramel Macchiato,130.0,210.0,245.0,80.0,115.0,38.10,46.94
3,Matcha Latte,130.0,190.0,225.0,60.0,95.0,31.58,42.22
4,Mocha,150.0,205.0,220.0,55.0,70.0,26.83,31.82


In [6]:
price_change_scenarios = pd.DataFrame({
    "price_change_pct": [
        0,
        3,
        5,
        7,
        10
    ]
})

bfc_current_average = (
    pricing_headroom[
        "bfc_price"
    ].mean()
)

price_change_scenarios[
    "illustrative_avg_price_php"
] = (
    bfc_current_average
    * (
        1
        + price_change_scenarios[
            "price_change_pct"
        ] / 100
    )
)

price_change_scenarios.round(2)

,price_change_pct,illustrative_avg_price_php
0,0,138.00
1,3,142.14
2,5,144.90
3,7,147.66
4,10,151.80


In [7]:
price_change_scenarios[
    "required_volume_index_for_revenue_neutrality"
] = (
    1
    / (
        1
        + price_change_scenarios[
            "price_change_pct"
        ] / 100
    )
)

price_change_scenarios[
    "maximum_volume_decline_pct"
] = (
    1
    - price_change_scenarios[
        "required_volume_index_for_revenue_neutrality"
    ]
) * 100

price_change_scenarios.round(2)

,price_change_pct,illustrative_avg_price_php,required_volume_index_for_revenue_neutrality,maximum_volume_decline_pct
0,0,138.00,1.00,0.00
1,3,142.14,0.97,2.91
2,5,144.90,0.95,4.76
3,7,147.66,0.93,6.54
4,10,151.80,0.91,9.09


In [8]:
price_changes = [
    0.00,
    0.03,
    0.05,
    0.07,
    0.10
]

volume_changes = [
    0.00,
    -0.02,
    -0.05,
    -0.08,
    -0.10
]

scenario_rows = []

for price_change in price_changes:

    for volume_change in volume_changes:

        revenue_index = (
            (1 + price_change)
            * (1 + volume_change)
        )

        scenario_rows.append({
            "price_change_pct":
                price_change * 100,

            "volume_change_pct":
                volume_change * 100,

            "revenue_index":
                revenue_index,

            "revenue_change_pct":
                (
                    revenue_index - 1
                ) * 100
        })

pricing_scenario_matrix = pd.DataFrame(
    scenario_rows
)

pricing_scenario_matrix.round(2)

,price_change_pct,volume_change_pct,revenue_index,revenue_change_pct
0,0.0,0.0,1.00,0.00
1,0.0,-2.0,0.98,-2.00
2,0.0,-5.0,0.95,-5.00
3,0.0,-8.0,0.92,-8.00
4,0.0,-10.0,0.90,-10.00
5,3.0,0.0,1.03,3.00
6,3.0,-2.0,1.01,0.94
7,3.0,-5.0,0.98,-2.15
8,3.0,-8.0,0.95,-5.24
9,3.0,-10.0,0.93,-7.30


In [9]:
pricing_scenario_pivot = (
    pricing_scenario_matrix
    .pivot(
        index="volume_change_pct",
        columns="price_change_pct",
        values="revenue_change_pct"
    )
    .sort_index(
        ascending=False
    )
)

pricing_scenario_pivot.round(2)

price_change_pct,0.0,3.0,5.0,7.0,10.0
volume_change_pct,,,,,
0.0,0.0,3.00,5.00,7.00,10.0
-2.0,-2.0,0.94,2.90,4.86,7.8
-5.0,-5.0,-2.15,-0.25,1.65,4.5
-8.0,-8.0,-5.24,-3.40,-1.56,1.2
-10.0,-10.0,-7.30,-5.50,-3.70,-1.0


## Pricing Headroom Interpretation

The matched competitive analysis shows that But First, Coffee occupies a consistently lower observed price position across the five directly comparable product families.

This creates potential pricing headroom, but competitive price gaps alone do not justify a broad price increase.

Revenue impact depends on the interaction between price and customer volume:

\[
Revenue\ Change
=
(1 + Price\ Change)
\times
(1 + Volume\ Change)
-1
\]

For example, under the illustrative scenarios:

- a 3% price increase can tolerate approximately a 2.91% volume decline before revenue falls below the current level;
- a 5% increase can tolerate approximately a 4.76% decline;
- a 7% increase can tolerate approximately a 6.54% decline; and
- a 10% increase can tolerate approximately a 9.09% decline.

These thresholds refer to revenue neutrality only. They do not estimate profit neutrality because product-level variable costs and contribution margins are unavailable.

The appropriate management implication is therefore not an immediate across-the-board price increase. Instead, the evidence supports testing selective pricing opportunities on suitable products, branches, channels, or customer segments while monitoring transaction volume, product mix, and customer response.

In [10]:
bfc_menu = (
    menu_prices[
        menu_prices["brand"]
        == "But First, Coffee"
    ]
    .copy()
)

print(
    "BFC menu observations:",
    len(bfc_menu)
)

print(
    "Observed promotional items:",
    bfc_menu["is_promo"].sum()
)

print(
    "Observed promotion coverage:",
    f"{bfc_menu['is_promo'].mean() * 100:.2f}%"
)

BFC menu observations: 50
Observed promotional items: 45
Observed promotion coverage: 90.00%


In [11]:
bfc_promotions = (
    bfc_menu[
        bfc_menu["is_promo"]
    ]
    .copy()
)

promotion_summary = pd.DataFrame({
    "metric": [
        "Promotional observations",
        "Mean discount %",
        "Median discount %",
        "Minimum discount %",
        "Maximum discount %"
    ],

    "value": [
        len(bfc_promotions),

        bfc_promotions[
            "discount_pct"
        ].mean(),

        bfc_promotions[
            "discount_pct"
        ].median(),

        bfc_promotions[
            "discount_pct"
        ].min(),

        bfc_promotions[
            "discount_pct"
        ].max()
    ]
})

promotion_summary.round(2)

,metric,value
0,Promotional observations,45.0
1,Mean discount %,20.0
2,Median discount %,20.0
3,Minimum discount %,20.0
4,Maximum discount %,20.0


## Promotion Revenue-Neutrality Framework

A discount reduces revenue per unit and therefore requires additional unit volume to maintain the same revenue level.

Let:

\[
P_0 = \text{Regular Price}
\]

\[
Q_0 = \text{Baseline Unit Volume}
\]

and let \(d\) represent the discount rate.

After discounting:

\[
P_1 = P_0(1-d)
\]

For revenue neutrality:

\[
P_0Q_0 = P_0(1-d)Q_1
\]

Therefore:

\[
\frac{Q_1}{Q_0}
=
\frac{1}{1-d}
\]

and the required percentage increase in unit volume is:

\[
Required\ Volume\ Increase
=
\left(
\frac{1}{1-d}-1
\right)
\times100
\]

This calculation measures revenue neutrality only. It does not measure contribution-margin neutrality because product-level variable costs are unavailable.

In [12]:
discount_scenarios = pd.DataFrame({
    "discount_pct": [
        5,
        10,
        15,
        20,
        25,
        30
    ]
})

discount_scenarios[
    "price_retained_pct"
] = (
    100
    - discount_scenarios[
        "discount_pct"
    ]
)

discount_scenarios[
    "required_volume_index"
] = (
    1
    / (
        1
        - discount_scenarios[
            "discount_pct"
        ] / 100
    )
)

discount_scenarios[
    "required_volume_increase_pct"
] = (
    discount_scenarios[
        "required_volume_index"
    ]
    - 1
) * 100

discount_scenarios.round(2)

,discount_pct,price_retained_pct,required_volume_index,required_volume_increase_pct
0,5,95,1.05,5.26
1,10,90,1.11,11.11
2,15,85,1.18,17.65
3,20,80,1.25,25.00
4,25,75,1.33,33.33
5,30,70,1.43,42.86


In [13]:
illustrative_regular_price = 150
illustrative_discount = 0.20
baseline_units = 100

promo_price = (
    illustrative_regular_price
    * (1 - illustrative_discount)
)

baseline_revenue = (
    illustrative_regular_price
    * baseline_units
)

required_promo_units = (
    baseline_revenue
    / promo_price
)

required_extra_units = (
    required_promo_units
    - baseline_units
)

print(
    "Illustrative regular price:",
    f"₱{illustrative_regular_price:.2f}"
)

print(
    "20% promotional price:",
    f"₱{promo_price:.2f}"
)

print(
    "Baseline units:",
    baseline_units
)

print(
    "Baseline revenue:",
    f"₱{baseline_revenue:,.2f}"
)

print(
    "Units required at promo price:",
    f"{required_promo_units:.0f}"
)

print(
    "Additional units required:",
    f"{required_extra_units:.0f}"
)

Illustrative regular price: ₱150.00
20% promotional price: ₱120.00
Baseline units: 100
Baseline revenue: ₱15,000.00
Units required at promo price: 125
Additional units required: 25


In [14]:
promo_volume_changes = [
    0,
    10,
    20,
    25,
    30,
    40,
    50
]

promo_response = pd.DataFrame({
    "volume_increase_pct":
        promo_volume_changes
})

discount_rate = 0.20

promo_response[
    "revenue_index"
] = (
    (1 - discount_rate)
    * (
        1
        + promo_response[
            "volume_increase_pct"
        ] / 100
    )
)

promo_response[
    "revenue_change_pct"
] = (
    promo_response[
        "revenue_index"
    ]
    - 1
) * 100

promo_response.round(2)

,volume_increase_pct,revenue_index,revenue_change_pct
0,0,0.80,-20.0
1,10,0.88,-12.0
2,20,0.96,-4.0
3,25,1.00,0.0
4,30,1.04,4.0
5,40,1.12,12.0
6,50,1.20,20.0


## Promotion Economics Interpretation

In the collected menu snapshot, 45 of 50 But First, Coffee menu observations carried an observed promotional price.

This represents 90% observed promotion coverage in the collected snapshot and should not be interpreted as evidence that 90% of the full menu is continuously discounted.

The observed promotional discount depth was approximately 20%.

At a 20% discount:

\[
Price\ Retained = 80\%
\]

and revenue neutrality requires:

\[
\frac{1}{0.80}=1.25
\]

or approximately:

\[
25\%
\]

additional unit volume.

For example, an item selling 100 units at ₱150 generates ₱15,000 in revenue. At a 20% promotional price of ₱120, the item would need to sell 125 units to generate the same ₱15,000.

Therefore, promotion performance should be evaluated against incremental volume rather than promotional sales volume alone.

A promotion that increases unit sales by less than 25% would reduce product revenue relative to the baseline under this simplified 20% discount scenario. A volume increase greater than 25% would increase revenue.

However, revenue neutrality is not equivalent to profit neutrality. With access to product-level variable costs, the appropriate next step would be to calculate the incremental unit volume required to preserve contribution margin.

The management opportunity is therefore to evaluate promotions using incremental economics: baseline units, promotional units, discount depth, incremental revenue, product mix, attachment effects, and contribution margin.

In [15]:
bfc_product_mix = (
    bfc_menu[
        "product_class"
    ]
    .value_counts()
    .rename_axis(
        "product_class"
    )
    .reset_index(
        name="menu_items"
    )
)

bfc_product_mix[
    "menu_share_pct"
] = (
    bfc_product_mix[
        "menu_items"
    ]
    / len(bfc_menu)
    * 100
)

bfc_product_mix.round(2)

,product_class,menu_items,menu_share_pct
0,Beverage,45,90.0
1,Food,3,6.0
2,Add-on,2,4.0


In [16]:
competitive_product_mix = (
    menu_prices
    .groupby([
        "brand",
        "product_class"
    ])
    .size()
    .rename("menu_items")
    .reset_index()
)

competitive_product_mix[
    "within_brand_share_pct"
] = (
    competitive_product_mix[
        "menu_items"
    ]
    / competitive_product_mix
        .groupby("brand")[
            "menu_items"
        ]
        .transform("sum")
    * 100
)

competitive_product_mix.round(2)

,brand,product_class,menu_items,within_brand_share_pct
0,"But First, Coffee",Add-on,2,4.0
1,"But First, Coffee",Beverage,45,90.0
2,"But First, Coffee",Food,3,6.0
3,Starbucks,Beverage,38,76.0
4,Starbucks,Food,12,24.0
5,The Coffee Bean & Tea Leaf,Beverage,39,78.0
6,The Coffee Bean & Tea Leaf,Food,11,22.0


In [17]:
bfc_class_pricing = (
    bfc_menu
    .groupby(
        "product_class"
    )
    .agg(
        menu_items=(
            "product",
            "count"
        ),
        mean_regular_price_php=(
            "regular_price_php",
            "mean"
        ),
        median_regular_price_php=(
            "regular_price_php",
            "median"
        ),
        min_regular_price_php=(
            "regular_price_php",
            "min"
        ),
        max_regular_price_php=(
            "regular_price_php",
            "max"
        )
    )
    .reset_index()
)

bfc_class_pricing.round(2)

,product_class,menu_items,mean_regular_price_php,median_regular_price_php,min_regular_price_php,max_regular_price_php
0,Add-on,2,30.00,30.0,30,30
1,Beverage,45,141.91,130.0,69,219
2,Food,3,91.67,75.0,60,140


## Basket-Building Revenue Framework

Revenue growth does not require every opportunity to come from higher beverage prices.

Average Transaction Value can also increase when customers purchase additional items:

\[
ATV
=
Average\ Selling\ Price
\times
Items\ per\ Transaction
\]

Potential basket-building mechanisms include:

- food attachment to beverage purchases;
- paid add-ons;
- size upgrades;
- premium product variants;
- complementary product bundles; and
- occasion-based combinations.

Without transaction-level POS data, the current analysis cannot estimate actual attachment rates, items per transaction, or Average Transaction Value.

Instead, scenario analysis is used to demonstrate how incremental attachment could affect transaction value under clearly stated assumptions.

In [18]:
base_beverage_price = 130
incremental_item_value = 50

attachment_rates = [
    0,
    5,
    10,
    15,
    20,
    25,
    30
]

attachment_scenarios = pd.DataFrame({
    "attachment_rate_pct":
        attachment_rates
})

attachment_scenarios[
    "base_transaction_value_php"
] = base_beverage_price

attachment_scenarios[
    "expected_incremental_value_php"
] = (
    incremental_item_value
    * attachment_scenarios[
        "attachment_rate_pct"
    ]
    / 100
)

attachment_scenarios[
    "illustrative_atv_php"
] = (
    attachment_scenarios[
        "base_transaction_value_php"
    ]
    + attachment_scenarios[
        "expected_incremental_value_php"
    ]
)

attachment_scenarios[
    "atv_increase_pct"
] = (
    (
        attachment_scenarios[
            "illustrative_atv_php"
        ]
        / base_beverage_price
    )
    - 1
) * 100

attachment_scenarios.round(2)

,attachment_rate_pct,base_transaction_value_php,expected_incremental_value_php,illustrative_atv_php,atv_increase_pct
0,0,130,0.0,130.0,0.00
1,5,130,2.5,132.5,1.92
2,10,130,5.0,135.0,3.85
3,15,130,7.5,137.5,5.77
4,20,130,10.0,140.0,7.69
5,25,130,12.5,142.5,9.62
6,30,130,15.0,145.0,11.54


In [19]:
attachment_scenarios[
    "revenue_index_if_transactions_constant"
] = (
    attachment_scenarios[
        "illustrative_atv_php"
    ]
    / base_beverage_price
)

attachment_scenarios[
    "revenue_change_pct_if_transactions_constant"
] = (
    attachment_scenarios[
        "revenue_index_if_transactions_constant"
    ]
    - 1
) * 100

attachment_scenarios.round(2)

,attachment_rate_pct,base_transaction_value_php,expected_incremental_value_php,illustrative_atv_php,atv_increase_pct,revenue_index_if_transactions_constant,revenue_change_pct_if_transactions_constant
0,0,130,0.0,130.0,0.00,1.00,0.00
1,5,130,2.5,132.5,1.92,1.02,1.92
2,10,130,5.0,135.0,3.85,1.04,3.85
3,15,130,7.5,137.5,5.77,1.06,5.77
4,20,130,10.0,140.0,7.69,1.08,7.69
5,25,130,12.5,142.5,9.62,1.10,9.62
6,30,130,15.0,145.0,11.54,1.12,11.54


## Basket Growth Versus Price Growth

Selective price increases and basket-building influence revenue through different mechanisms.

A price increase raises revenue per existing item but may create price-elasticity risk if customers reduce purchase frequency, switch products, or leave the brand.

Basket-building instead attempts to increase transaction value through additional customer purchases while preserving the entry price of the core beverage.

For a brand whose observed competitive position emphasizes affordability, basket-building may therefore provide a complementary revenue lever to selective pricing.

However, the external data do not establish current attachment rates, customer willingness to add products, or product-level contribution margins.

The appropriate internal analysis would measure:

- current items per transaction;
- beverage-to-food attachment rate;
- add-on attachment rate;
- size-upgrade rate;
- bundle conversion;
- incremental revenue per transaction;
- incremental contribution margin; and
- whether attachment changes transaction frequency.

The scenario analysis therefore demonstrates the revenue mechanism rather than forecasting actual BFC performance.

In [20]:
bfc_reviews = (
    reviews[
        reviews["brand"]
        == "But First, Coffee"
    ]
    .copy()
)

dimension_columns = {
    "Product Quality":
        "dimension_product_quality",

    "Order Execution":
        "dimension_order_execution",

    "Customization & Add-ons":
        "dimension_customization_addons",

    "Packaging":
        "dimension_packaging",

    "Value & Portion":
        "dimension_value_portion",

    "Service & Delivery":
        "dimension_service_delivery",

    "Availability & Consistency":
        "dimension_availability_consistency",

    "Loyalty Experience":
        "dimension_loyalty_experience"
}

bfc_dimension_rows = []

for dimension, column in dimension_columns.items():

    mentions = (
        bfc_reviews[column]
        .fillna(False)
        .astype(bool)
        .sum()
    )

    prevalence = (
        mentions
        / len(bfc_reviews)
        * 100
    )

    bfc_dimension_rows.append({
        "dimension": dimension,
        "mentions": mentions,
        "prevalence_pct": prevalence
    })

bfc_dimension_prevalence = pd.DataFrame(
    bfc_dimension_rows
).sort_values(
    "prevalence_pct",
    ascending=False
)

bfc_dimension_prevalence.round(2)

,dimension,mentions,prevalence_pct
0,Product Quality,52,50.00
3,Packaging,20,19.23
4,Value & Portion,20,19.23
1,Order Execution,16,15.38
6,Availability & Consistency,14,13.46
2,Customization & Add-ons,13,12.50
5,Service & Delivery,8,7.69
7,Loyalty Experience,4,3.85


In [21]:
bfc_theme_columns = {
    "Taste": "theme_taste",
    "Packaging": "theme_packaging",
    "Missing Item": "theme_missing_item",
    "Customization": "theme_customization",
    "Order Accuracy": "theme_order_accuracy",
    "Value": "theme_value",
    "Portion": "theme_portion",
    "Size Accuracy": "theme_size_accuracy",
    "Add-on": "theme_add_on",
    "Consistency": "theme_consistency",
    "Service": "theme_service",
    "Speed": "theme_speed",
    "Availability": "theme_availability"
}

bfc_theme_rows = []

for theme, column in bfc_theme_columns.items():

    mentions = (
        bfc_reviews[column]
        .fillna(False)
        .astype(bool)
        .sum()
    )

    prevalence = (
        mentions
        / len(bfc_reviews)
        * 100
    )

    bfc_theme_rows.append({
        "theme": theme,
        "mentions": mentions,
        "prevalence_pct": prevalence
    })

bfc_theme_prevalence = (
    pd.DataFrame(
        bfc_theme_rows
    )
    .sort_values(
        "prevalence_pct",
        ascending=False
    )
)

bfc_theme_prevalence.round(2)

,theme,mentions,prevalence_pct
0,Taste,49,47.12
1,Packaging,20,19.23
5,Value,16,15.38
9,Consistency,14,13.46
4,Order Accuracy,11,10.58
6,Portion,10,9.62
3,Customization,9,8.65
10,Service,6,5.77
2,Missing Item,4,3.85
8,Add-on,4,3.85


## Customer Experience as a Revenue-Risk Mechanism

Revenue management depends not only on price and demand generation but also on successful transaction execution and customer retention.

The review evidence cannot quantify actual lost revenue. However, several observed customer-experience themes can be mapped to plausible revenue mechanisms.

### Product Quality

Taste and consistency may influence perceived value, repeat purchase, and willingness to pay.

### Order Execution

Missing items, incorrect orders, and size inaccuracies may create:

- refunds or replacements;
- product waste;
- service recovery costs;
- lower perceived transaction value; and
- potential repeat-purchase risk.

### Customization & Add-ons

Failures in paid customization or add-on execution may reduce the value of upselling strategies and weaken customer confidence in premium additions.

### Packaging

Packaging problems may be particularly relevant to delivery-channel economics because product quality must survive the fulfillment process.

### Availability & Consistency

Unavailable products or inconsistent execution may reduce conversion and limit the effectiveness of pricing, promotion, and basket-building initiatives.

These mechanisms are commercially plausible interpretations rather than measured causal effects. Transaction-level POS, refund, cancellation, complaint, and repeat-purchase data would be required to quantify their actual revenue impact.

In [22]:
revenue_risk_map = pd.DataFrame({
    "evidence_area": [
        "Product Quality",
        "Order Execution",
        "Customization & Add-ons",
        "Packaging",
        "Availability & Consistency"
    ],

    "observed_signal": [
        "Taste appears prominently in review evidence",
        "Missing-item and order-accuracy themes observed",
        "Customization and add-on themes observed",
        "Packaging appears in customer-experience evidence",
        "Availability and consistency themes observed"
    ],

    "potential_revenue_mechanism": [
        "Repeat purchase, perceived value, willingness to pay",
        "Refunds, replacements, waste, retention risk",
        "Upsell conversion and paid-add-on value",
        "Delivery experience and service recovery",
        "Conversion, repeat purchase and product availability"
    ],

    "internal_metric_needed": [
        "SKU repeat rate, ratings, complaints, repurchase",
        "Error rate, refund rate, remake cost, cancellations",
        "Add-on attachment, error rate, incremental margin",
        "Packaging complaints, refunds, delivery ratings",
        "Stock-outs, lost sales, availability by SKU"
    ]
})

revenue_risk_map

,evidence_area,observed_signal,potential_revenue_mechanism,internal_metric_needed
0,Product Quality,Taste appears prominently in review evidence,"Repeat purchase, perceived value, willingness ...","SKU repeat rate, ratings, complaints, repurchase"
1,Order Execution,Missing-item and order-accuracy themes observed,"Refunds, replacements, waste, retention risk","Error rate, refund rate, remake cost, cancella..."
2,Customization & Add-ons,Customization and add-on themes observed,Upsell conversion and paid-add-on value,"Add-on attachment, error rate, incremental margin"
3,Packaging,Packaging appears in customer-experience evidence,Delivery experience and service recovery,"Packaging complaints, refunds, delivery ratings"
4,Availability & Consistency,Availability and consistency themes observed,"Conversion, repeat purchase and product availa...","Stock-outs, lost sales, availability by SKU"


## Revenue-Management Interpretation

The customer-experience evidence suggests that revenue initiatives should be evaluated together with operational execution.

For example, increasing add-on attachment may raise Average Transaction Value in principle, but the commercial benefit depends on accurate fulfillment of those add-ons.

Similarly, selective price increases may be more sustainable where product quality and consistency support perceived value.

This creates an important sequence for revenue management:

\[
Operational\ Reliability
\rightarrow
Customer\ Value
\rightarrow
Revenue\ Lever
\rightarrow
Revenue\ Outcome
\]

The external evidence does not quantify the financial effect of operational failures. It identifies where internal transaction and operational data should be connected to the revenue model.

A stronger internal model would integrate POS transactions with:

- refunds and cancellations;
- remakes and order errors;
- product availability;
- add-on attachment;
- customer complaints;
- repeat purchase;
- promotional exposure; and
- contribution margin.

This would allow management to distinguish revenue growth generated through sustainable customer value from revenue that may be offset by execution failures or service recovery.

In [23]:
channels[
    [
        "brand",
        "type",
        "partner_or_channel",
        "current_evidence",
        "revenue_relevance"
    ]
].sort_values(
    [
        "brand",
        "type"
    ]
)

,brand,type,partner_or_channel,current_evidence,revenue_relevance
2,"But First, Coffee",Delivery marketplace,Foodpanda,"Multiple current branches, menus, promotions, ...",Delivery revenue / reach
3,"But First, Coffee",Franchise,BFC franchise model,Official brand positions itself as a cafe fran...,Expansion / capital-light footprint
1,"But First, Coffee",Payments + rewards partnership,BPI VYBE / QR Ph,2026 promo across 237+ participating branches;...,"Digital payment adoption, acquisition, promotion"
0,"But First, Coffee",Store locator,Official BFC store locator,Official site has nationwide store locator,Physical access / footprint
4,Starbucks,Delivery,GrabFood,Official Starbucks delivery page confirms Grab...,Delivery revenue / reach
5,Starbucks,Delivery,Foodpanda,Official Starbucks delivery page confirms food...,Delivery revenue / reach
6,Starbucks,Delivery,Pick.A.Roo,Official Starbucks delivery page confirms Pick...,Delivery revenue / reach
9,Starbucks,Gifting,GrabGifts,Starbucks x Grab partnership supports digital ...,Incremental occasions / gifting revenue
7,Starbucks,Loyalty integration,Starbucks Rewards + Grab,Members can earn Stars on eligible GrabFood St...,Retention / repeat purchase
8,Starbucks,Mobile ordering,Starbucks PH app / Mobile Order & Pay,"Official order page describes order ahead, pic...",Convenience / frequency / queue reduction


In [24]:
channel_count_summary = (
    channels
    .groupby("brand")
    .agg(
        observed_channel_records=(
            "partner_or_channel",
            "count"
        ),
        observed_channel_types=(
            "type",
            "nunique"
        )
    )
    .reset_index()
)

channel_count_summary

,brand,observed_channel_records,observed_channel_types
0,"But First, Coffee",4,4
1,Starbucks,6,4
2,The Coffee Bean & Tea Leaf,6,4


In [25]:
channel_type_matrix = (
    channels
    .groupby([
        "brand",
        "type"
    ])
    .size()
    .unstack(
        fill_value=0
    )
)

channel_type_matrix

type,Delivery,Delivery marketplace,E-commerce,Franchise,Gifting,Loyalty,Loyalty / campaign,Loyalty integration,Mobile ordering,Payments + rewards partnership,Store locator
brand,,,,,,,,,,,
"But First, Coffee",0,1,0,1,0,0,0,0,0,1,1
Starbucks,3,0,0,0,1,0,0,1,1,0,0
The Coffee Bean & Tea Leaf,3,0,1,0,0,1,1,0,0,0,0


## Commercial Access as a Revenue Lever

Revenue opportunity depends not only on menu pricing and customer basket size but also on the number and quality of opportunities customers have to transact with the brand.

Commercial access can influence revenue through mechanisms such as:

\[
Customer\ Access
\rightarrow
Transaction\ Opportunity
\rightarrow
Transaction\ Volume
\rightarrow
Revenue
\]

Observable channel evidence may include:

- delivery platforms;
- digital ordering;
- loyalty programs;
- payment partnerships;
- e-commerce;
- store-location access;
- gifting;
- franchise networks; and
- promotional partnerships.

However, the number of observed channels should not be interpreted as channel performance.

A brand with more observable channels does not necessarily generate more revenue because channel economics depend on customer adoption, transaction frequency, commissions, discounts, acquisition cost, retention, and contribution margin.

The external evidence is therefore used to identify commercial-access opportunities rather than to rank channel profitability.

In [26]:
revenue_lever_framework = pd.DataFrame({
    "revenue_lever": [
        "Selective Pricing",
        "Promotion Optimization",
        "Basket Building",
        "Operational Execution",
        "Commercial Access"
    ],

    "revenue_mechanism": [
        "Increase average selling price",
        "Generate incremental demand efficiently",
        "Increase items and value per transaction",
        "Protect conversion and repeat-purchase value",
        "Increase opportunities to transact"
    ],

    "external_evidence": [
        (
            "BFC was lowest-priced across all "
            "5 matched product families"
        ),
        (
            "45 of 50 observed BFC menu items "
            "carried a 20% promotional price"
        ),
        (
            "Collected BFC menu sample was "
            "90% beverage, 6% food and 4% add-on"
        ),
        (
            "Product quality, packaging, value, "
            "order execution and consistency appeared "
            "in BFC review evidence"
        ),
        (
            "Multiple customer-access and partnership "
            "channels observed across competitors"
        )
    ],

    "management_question": [
        (
            "Which products can sustain selective "
            "price tests without materially reducing volume?"
        ),
        (
            "Do promotions generate sufficient incremental "
            "units and contribution margin?"
        ),
        (
            "Can food, add-ons, upgrades or bundles "
            "increase ATV?"
        ),
        (
            "Where do execution failures create refunds, "
            "waste or repeat-purchase risk?"
        ),
        (
            "Which channels create incremental profitable "
            "transactions?"
        )
    ],

    "internal_data_required": [
        (
            "SKU volume, elasticity, mix, competitor tracking, "
            "variable cost"
        ),
        (
            "Baseline and promo units, promo exposure, "
            "variable cost, contribution margin"
        ),
        (
            "Transaction baskets, attachment rates, "
            "items per transaction, margin"
        ),
        (
            "Refunds, remakes, complaints, cancellations, "
            "repeat purchase"
        ),
        (
            "Channel transactions, commission, CAC, "
            "repeat rate, contribution margin"
        )
    ]
})

revenue_lever_framework

,revenue_lever,revenue_mechanism,external_evidence,management_question,internal_data_required
0,Selective Pricing,Increase average selling price,BFC was lowest-priced across all 5 matched pro...,Which products can sustain selective price tes...,"SKU volume, elasticity, mix, competitor tracki..."
1,Promotion Optimization,Generate incremental demand efficiently,45 of 50 observed BFC menu items carried a 20%...,Do promotions generate sufficient incremental ...,"Baseline and promo units, promo exposure, vari..."
2,Basket Building,Increase items and value per transaction,"Collected BFC menu sample was 90% beverage, 6%...","Can food, add-ons, upgrades or bundles increas...","Transaction baskets, attachment rates, items p..."
3,Operational Execution,Protect conversion and repeat-purchase value,"Product quality, packaging, value, order execu...","Where do execution failures create refunds, wa...","Refunds, remakes, complaints, cancellations, r..."
4,Commercial Access,Increase opportunities to transact,Multiple customer-access and partnership chann...,Which channels create incremental profitable t...,"Channel transactions, commission, CAC, repeat ..."


In [27]:
evidence_hierarchy = pd.DataFrame({
    "evidence_level": [
        "Statistical evidence",
        "Observed quantitative evidence",
        "Descriptive customer evidence",
        "Scenario analysis",
        "Management hypothesis"
    ],

    "example": [
        (
            "Friedman Q(2)=10.000, p=0.006738 "
            "across matched prices"
        ),
        (
            "45/50 observed BFC menu observations "
            "had a 20% promotional price"
        ),
        (
            "BFC review themes and management dimensions"
        ),
        (
            "Price-volume, promotion and attachment scenarios"
        ),
        (
            "Selective pricing, basket expansion and "
            "channel opportunities requiring internal testing"
        )
    ],

    "appropriate_use": [
        "Support the existence of an observed systematic pattern",
        "Describe the collected external evidence",
        "Identify customer-experience signals",
        "Illustrate commercial economics under assumptions",
        "Define what management should test internally"
    ]
})

evidence_hierarchy

,evidence_level,example,appropriate_use
0,Statistical evidence,"Friedman Q(2)=10.000, p=0.006738 across matche...",Support the existence of an observed systemati...
1,Observed quantitative evidence,45/50 observed BFC menu observations had a 20%...,Describe the collected external evidence
2,Descriptive customer evidence,BFC review themes and management dimensions,Identify customer-experience signals
3,Scenario analysis,"Price-volume, promotion and attachment scenarios",Illustrate commercial economics under assumptions
4,Management hypothesis,"Selective pricing, basket expansion and channe...",Define what management should test internally


In [28]:
channel_capability_map = {
    "Delivery marketplace": "Delivery",
    "Delivery": "Delivery",
    "Franchise": "Physical Expansion",
    "Payments + rewards partnership": "Payments & Partnerships",
    "Store locator": "Physical Access",
    "Gifting": "Gifting",
    "Loyalty integration": "Loyalty",
    "Mobile ordering": "Digital Ordering",
    "E-commerce": "E-commerce",
    "Loyalty": "Loyalty",
    "Loyalty / campaign": "Loyalty"
}

channels[
    "channel_capability"
] = (
    channels["type"]
    .map(channel_capability_map)
)

channel_capability_matrix = (
    channels
    .assign(
        observed=1
    )
    .pivot_table(
        index="brand",
        columns="channel_capability",
        values="observed",
        aggfunc="max",
        fill_value=0
    )
)

channel_capability_matrix

channel_capability,Delivery,Digital Ordering,E-commerce,Gifting,Loyalty,Payments & Partnerships,Physical Access,Physical Expansion
brand,,,,,,,,
"But First, Coffee",1,0,0,0,0,1,1,1
Starbucks,1,1,0,1,1,0,0,0
The Coffee Bean & Tea Leaf,1,0,1,0,1,0,0,0


In [29]:
channel_opportunity_framework = pd.DataFrame({
    "capability": [
        "Multi-platform Delivery",
        "Digital Ordering",
        "Loyalty / Retention",
        "Gifting",
        "E-commerce",
        "Payments & Partnerships",
        "Physical Expansion"
    ],

    "competitive_evidence": [
        "Starbucks and CBTL observed on 3 delivery platforms",
        "Starbucks Mobile Order & Pay observed",
        "Starbucks and CBTL loyalty mechanisms observed",
        "Starbucks GrabGifts observed",
        "CBTL marketplace e-commerce observed",
        "BFC BPI VYBE / QR Ph partnership observed",
        "BFC franchise model and store access observed"
    ],

    "potential_revenue_mechanism": [
        "Customer reach and transaction accessibility",
        "Convenience, frequency and queue reduction",
        "Repeat purchase and retention",
        "Incremental occasions and prepaid demand",
        "Non-store and merchandise revenue",
        "Acquisition, payment adoption and promotion",
        "Geographic reach and transaction capacity"
    ],

    "internal_validation_needed": [
        "Incremental orders, commissions, cannibalization, margin",
        "Adoption, order frequency, operational capacity, margin",
        "Repeat rate, redemption economics, retention lift",
        "Incremental occasions, redemption, breakage, margin",
        "Demand, fulfillment cost, incremental contribution",
        "Acquisition, repeat rate, promo cost, contribution",
        "Unit economics, store productivity, payback period"
    ]
})

channel_opportunity_framework

,capability,competitive_evidence,potential_revenue_mechanism,internal_validation_needed
0,Multi-platform Delivery,Starbucks and CBTL observed on 3 delivery plat...,Customer reach and transaction accessibility,"Incremental orders, commissions, cannibalizati..."
1,Digital Ordering,Starbucks Mobile Order & Pay observed,"Convenience, frequency and queue reduction","Adoption, order frequency, operational capacit..."
2,Loyalty / Retention,Starbucks and CBTL loyalty mechanisms observed,Repeat purchase and retention,"Repeat rate, redemption economics, retention lift"
3,Gifting,Starbucks GrabGifts observed,Incremental occasions and prepaid demand,"Incremental occasions, redemption, breakage, m..."
4,E-commerce,CBTL marketplace e-commerce observed,Non-store and merchandise revenue,"Demand, fulfillment cost, incremental contribu..."
5,Payments & Partnerships,BFC BPI VYBE / QR Ph partnership observed,"Acquisition, payment adoption and promotion","Acquisition, repeat rate, promo cost, contribu..."
6,Physical Expansion,BFC franchise model and store access observed,Geographic reach and transaction capacity,"Unit economics, store productivity, payback pe..."


## Commercial Access Interpretation

The collected evidence identifies different commercial-access models across the three brands.

But First, Coffee has observable evidence of delivery-marketplace access, a franchise model, a digital-payment and rewards partnership, and nationwide store-location access.

Starbucks additionally shows observable evidence of multi-platform delivery, mobile ordering, loyalty integration, and digital gifting.

The Coffee Bean & Tea Leaf shows observable evidence of multi-platform delivery, loyalty mechanisms, e-commerce, and campaign-based engagement.

These differences do not establish that competitor channels are more profitable or that But First, Coffee should replicate them.

Instead, they identify capabilities that can be evaluated using channel-level economics.

The appropriate management question is:

> Which additional customer-access mechanisms can generate incremental transactions or repeat purchases at an acceptable acquisition, commission, fulfillment, and contribution-margin cost?

Channel expansion should therefore be evaluated on incremental economics rather than channel count alone.

In [30]:
revenue_opportunity_matrix = pd.DataFrame({
    "opportunity": [
        "Promotion Optimization",
        "Operational Execution",
        "Basket Building",
        "Selective Pricing",
        "Commercial Access"
    ],

    "revenue_driver": [
        "Revenue efficiency",
        "Transaction protection",
        "Average Transaction Value",
        "Average Selling Price",
        "Transaction Volume"
    ],

    "external_evidence": [
        (
            "45/50 observed BFC menu observations "
            "carried a 20% promotional price"
        ),
        (
            "Product quality, packaging, value, "
            "order execution and consistency appear "
            "in BFC review evidence"
        ),
        (
            "Collected BFC menu sample is "
            "90% beverage, 6% food and 4% add-on"
        ),
        (
            "BFC was lowest-priced across all "
            "5 matched product families"
        ),
        (
            "Competitor evidence includes multi-platform "
            "delivery, loyalty, digital ordering, gifting "
            "and e-commerce capabilities"
        )
    ],

    "commercial_hypothesis": [
        (
            "Some promotions may surrender more revenue "
            "than the incremental demand they generate"
        ),
        (
            "Execution improvements may protect conversion, "
            "repeat purchase and revenue initiatives"
        ),
        (
            "Food, add-ons, upgrades and bundles may "
            "increase transaction value"
        ),
        (
            "Selected products may have room for controlled "
            "price testing while preserving affordability"
        ),
        (
            "Additional access capabilities may create "
            "incremental transaction occasions"
        )
    ],

    "recommended_management_test": [
        (
            "Measure incremental units and contribution "
            "margin against non-promotional baseline"
        ),
        (
            "Connect order errors, refunds, remakes and "
            "complaints with transaction and repeat data"
        ),
        (
            "Test food/add-on attachment and bundles "
            "against ATV and contribution margin"
        ),
        (
            "Run controlled SKU/branch price tests and "
            "measure elasticity, mix and retention"
        ),
        (
            "Pilot selected channels and measure incremental "
            "orders after commissions and cannibalization"
        )
    ],

    "decision_metric": [
        "Incremental contribution margin",
        "Revenue leakage + repeat-purchase impact",
        "ATV + contribution margin per transaction",
        "Revenue and contribution after volume response",
        "Incremental contribution per channel"
    ],

    "evidence_status": [
        "Strong external signal; internal economics required",
        "Descriptive signal; internal quantification required",
        "Scenario-supported hypothesis",
        "Statistically supported positioning; test required",
        "Competitive capability hypothesis"
    ]
})

revenue_opportunity_matrix

,opportunity,revenue_driver,external_evidence,commercial_hypothesis,recommended_management_test,decision_metric,evidence_status
0,Promotion Optimization,Revenue efficiency,45/50 observed BFC menu observations carried a...,Some promotions may surrender more revenue tha...,Measure incremental units and contribution mar...,Incremental contribution margin,Strong external signal; internal economics req...
1,Operational Execution,Transaction protection,"Product quality, packaging, value, order execu...","Execution improvements may protect conversion,...","Connect order errors, refunds, remakes and com...",Revenue leakage + repeat-purchase impact,Descriptive signal; internal quantification re...
2,Basket Building,Average Transaction Value,"Collected BFC menu sample is 90% beverage, 6% ...","Food, add-ons, upgrades and bundles may increa...",Test food/add-on attachment and bundles agains...,ATV + contribution margin per transaction,Scenario-supported hypothesis
3,Selective Pricing,Average Selling Price,BFC was lowest-priced across all 5 matched pro...,Selected products may have room for controlled...,Run controlled SKU/branch price tests and meas...,Revenue and contribution after volume response,Statistically supported positioning; test requ...
4,Commercial Access,Transaction Volume,Competitor evidence includes multi-platform de...,Additional access capabilities may create incr...,Pilot selected channels and measure incrementa...,Incremental contribution per channel,Competitive capability hypothesis


# Integrated Revenue Opportunity

The external analysis identifies five commercially relevant revenue levers for But First, Coffee:

### 1. Promotion Optimization

The collected menu snapshot showed 45 of 50 BFC observations at a 20% promotional discount.

A 20% discount requires approximately 25% additional unit volume merely to preserve revenue before considering costs.

The management priority is therefore to measure promotion incrementality rather than promotional sales volume alone.

### 2. Operational Execution

BFC customer-review evidence frequently references product quality, packaging, value, order execution and consistency.

These signals cannot quantify lost revenue externally, but they identify operational areas that should be connected to refunds, remakes, cancellations, repeat purchase and contribution margin.

### 3. Basket Building

The collected BFC menu sample was heavily beverage-oriented.

Illustrative attachment scenarios demonstrate that increasing food, add-on, upgrade or bundle attachment can raise Average Transaction Value without necessarily increasing the entry price of the core beverage.

Actual opportunity depends on transaction-level attachment rates and contribution margins.

### 4. Selective Pricing

Across five directly matched product families, BFC occupied the lowest observed price position.

The Friedman test identified a significant systematic difference in price ranks:

\[
Q(2)=10.000,\quad p=0.006738
\]

with:

\[
W=1.00
\]

indicating complete rank consistency across the five matched families.

This supports selective pricing as a testable opportunity, not an automatic across-the-board price increase.

### 5. Commercial Access

Competitor evidence demonstrates additional transaction mechanisms including multi-platform delivery, digital ordering, loyalty integration, gifting and e-commerce.

These represent capabilities for evaluation rather than proof that more channels automatically create more profitable revenue.

---

## Management Principle

The central recommendation is not simply to raise prices.

Instead:

\[
\boxed{
Profitable\ Revenue\ Growth
=
Pricing\ Discipline
+
Promotion\ Efficiency
+
Basket\ Growth
+
Execution\ Reliability
+
Commercial\ Access
}
\]

Each lever should ultimately be evaluated using internal contribution-margin economics.

In [32]:
internal_data_roadmap = pd.DataFrame({
    "dataset": [
        "POS Transactions",
        "Product Cost",
        "Promotion History",
        "Order Operations",
        "Customer / Loyalty",
        "Channel Performance"
    ],

    "minimum_fields": [
        (
            "transaction_id, date, branch, SKU, "
            "quantity, selling_price, discount"
        ),
        (
            "SKU, ingredient cost, packaging cost, "
            "variable cost"
        ),
        (
            "SKU, campaign, start/end date, "
            "discount, exposure"
        ),
        (
            "transaction_id, error, remake, refund, "
            "cancellation, fulfillment time"
        ),
        (
            "customer_id, transactions, frequency, "
            "basket, loyalty activity"
        ),
        (
            "channel, transactions, revenue, commission, "
            "discount, fulfillment cost"
        )
    ],

    "revenue_analysis_enabled": [
        "ATV, items/transaction, mix, elasticity, branch productivity",
        "Contribution margin and menu engineering",
        "Incrementality and promotion ROI",
        "Revenue leakage and operational cost",
        "Retention, frequency and customer lifetime value",
        "Incremental channel economics"
    ]
})

internal_data_roadmap

,dataset,minimum_fields,revenue_analysis_enabled
0,POS Transactions,"transaction_id, date, branch, SKU, quantity, s...","ATV, items/transaction, mix, elasticity, branc..."
1,Product Cost,"SKU, ingredient cost, packaging cost, variable...",Contribution margin and menu engineering
2,Promotion History,"SKU, campaign, start/end date, discount, exposure",Incrementality and promotion ROI
3,Order Operations,"transaction_id, error, remake, refund, cancell...",Revenue leakage and operational cost
4,Customer / Loyalty,"customer_id, transactions, frequency, basket, ...","Retention, frequency and customer lifetime value"
5,Channel Performance,"channel, transactions, revenue, commission, di...",Incremental channel economics


In [33]:
pricing_headroom.to_csv(
    TABLES / "revenue_pricing_headroom.csv",
    index=False
)

price_change_scenarios.to_csv(
    TABLES / "revenue_price_change_scenarios.csv",
    index=False
)

pricing_scenario_matrix.to_csv(
    TABLES / "revenue_price_volume_scenarios.csv",
    index=False
)

discount_scenarios.to_csv(
    TABLES / "revenue_discount_scenarios.csv",
    index=False
)

promo_response.to_csv(
    TABLES / "revenue_promotion_response.csv",
    index=False
)

bfc_product_mix.to_csv(
    TABLES / "revenue_bfc_product_mix.csv",
    index=False
)

competitive_product_mix.to_csv(
    TABLES / "revenue_competitive_product_mix.csv",
    index=False
)

bfc_class_pricing.to_csv(
    TABLES / "revenue_bfc_class_pricing.csv",
    index=False
)

attachment_scenarios.to_csv(
    TABLES / "revenue_attachment_scenarios.csv",
    index=False
)

bfc_dimension_prevalence.to_csv(
    TABLES / "revenue_bfc_dimension_prevalence.csv",
    index=False
)

bfc_theme_prevalence.to_csv(
    TABLES / "revenue_bfc_theme_prevalence.csv",
    index=False
)

revenue_risk_map.to_csv(
    TABLES / "revenue_risk_map.csv",
    index=False
)

channel_capability_matrix.to_csv(
    TABLES / "revenue_channel_capability_matrix.csv"
)

channel_opportunity_framework.to_csv(
    TABLES / "revenue_channel_opportunity_framework.csv",
    index=False
)

revenue_lever_framework.to_csv(
    TABLES / "revenue_lever_framework.csv",
    index=False
)

revenue_opportunity_matrix.to_csv(
    TABLES / "revenue_opportunity_matrix.csv",
    index=False
)

internal_data_roadmap.to_csv(
    TABLES / "revenue_internal_data_roadmap.csv",
    index=False
)

print(
    "Notebook 05 revenue opportunity outputs "
    "exported successfully."
)

Notebook 05 revenue opportunity outputs exported successfully.


# Revenue Opportunity Analysis Conclusion

The external evidence suggests that But First, Coffee's revenue opportunity should not be reduced to a single pricing decision.

The brand occupies a consistently affordable observed position across the five matched product families, creating a basis for controlled selective-pricing tests. However, the impact of any price change depends on customer volume response and product mix.

Promotion economics represent another important area for internal validation. In the collected snapshot, 45 of 50 BFC menu observations carried a 20% promotional price. At this discount depth, unit volume must increase by approximately 25% merely to preserve revenue before considering costs.

Basket building provides an alternative route to revenue growth. Illustrative attachment scenarios show how additional food, add-ons, upgrades or bundles can increase Average Transaction Value while maintaining an affordable core beverage price.

Customer-experience evidence further indicates that revenue initiatives should be evaluated together with operational execution. Product quality, packaging, value, order execution and consistency appear prominently in BFC review evidence, but their financial effects require internal transaction and operational data for quantification.

Finally, competitor channel evidence identifies additional commercial-access mechanisms that BFC can evaluate, including multi-platform delivery, digital ordering, loyalty, gifting and e-commerce. These should be assessed through incremental channel economics rather than copied on the basis of competitor presence alone.

The resulting management framework is:

\[
Revenue
=
Transactions
\times
Average\ Transaction\ Value
\]

supported by five interconnected levers:

1. selective pricing;
2. promotion optimization;
3. basket building;
4. operational execution; and
5. commercial access.

Actual profitability cannot be estimated from the external dataset because product costs and contribution margins are unavailable.

The appropriate next step with internal company data would therefore be to connect transaction, product-cost, promotion, operational, customer and channel datasets to quantify contribution-margin opportunities and prioritize implementation.